## Using edgeR for DEG analysis between clusters -- comparing region_leiden clusters within capillary cells and using pseurod-bulk approach

In [ ]:
### identify cluster marker genes
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    library(harmony)
    library(reticulate)
    library(viridis)
    library(RColorBrewer)
    library(ComplexHeatmap)
    library(colorRamp2)
    library(edgeR)
})

# set large memory  
options(future.globals.maxSize = 120*1024^3)

## plotting parameters
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))

## set working dir
dir = "/scratch/users/aliu10/vascular"
setwd(dir)

In [ ]:
# obj_oi = readRDS("./Results/Revision_2/Endothelial.rds")
## Reloading 
# Convert h5ad to h5seurat
h5ad_file="./Results/Revision_2/Mural_temp_processed.h5ad"
obj_oi = schard::h5ad2seurat(h5ad_file)
print(obj_oi)

In [ ]:
## check cluster sizes and combine small clusters if needed
table(obj_oi$Cell_type,useNA = "always")

# table(obj_oi$region_cluster,useNA = "always")

## Check the UMAP of the mural cell types


In [ ]:
col1=c(
  "SMC_1" = "#6A0DAD",   # strong violet
  "SMC_2" = "#A26DC2",   # medium purple
  "SMC_3" = "#C29DFF",
  "Pericyte" = "#5A8E60"
)

In [ ]:
FeaturePlot(obj_oi,features="PCDH9",reduction = "umapharmony_",pt.size = 3,label = TRUE)

In [ ]:
options(repr.plot.height = 7,repr.plot.width = 8)
p = DimPlot(obj_oi,
    group.by = "Cell_type",
    cols = col1,
    reduction = "umapharmony_",
    pt.size = 1.5,
    label = TRUE,
    label.size = 10,
    repel = TRUE,
    raster = FALSE
    ) + umap_theme + NoLegend()


ggsave(filename = "./Results/Revision_2/Figures/Figure_4a_mural_cell_type_UMAP.pdf",
    patchwork::wrap_plots(p,ncol = 1),
      scale = 1, width = 8, height = 7)

In [ ]:
## exclude samples with less than 3 cells
meta = obj_oi@meta.data
mat = table(meta$sampleID)
sample_excluded = names(mat[mat<3])
length(sample_excluded)

obj_oi = subset(obj_oi,subset = !sampleID %in% sample_excluded)
obj_oi

In [ ]:
## further filter genes with less expression
min_features_pct =0.05
min_features_ncell = 20

##--- Get the expression summary ---##
# obj_oi <- JoinLayers(obj_oi)
# expr = LayerData(obj_oi, assay = "decontX", layer = "counts")
expr = LayerData(obj_oi, assay = "RNA", layer = "counts")

## calculate percentage of cells expressing each gene
genes.percent.expression <- rowMeans(expr>0)
# genes.percent.expression = rowSums(expr>0)

## select genes based on expression threshold
genes = names(genes.percent.expression[genes.percent.expression>min_features_pct])
# genes = names(genes.percent.expression[genes.percent.expression>min_features_ncell])

print(paste(length(genes),"genes will be tested."))

## Using wilcoxon rank sum test to identify cluster marker genes

In [ ]:
# Normalize the data
obj_oi <- NormalizeData(obj_oi)
Idents(obj_oi) = obj_oi$Cell_type
obj_oi

In [ ]:
#### code for identifying the cell type makers ####
# the parameters are based on https://www.cell.com/cell/fulltext/S0092-8674(23)00973-X
table(Idents(obj_oi))
# clean <- PrepSCTFindMarkers(clean)
# running differential analysis
all_markers_major <- FindAllMarkers(object = obj_oi,verbose = T,min.pct = 0.25,logfc.threshold = 0.25
#,test.use = "MAST" #might use MAST model
) 
write.csv(all_markers_major,file = "./Results/Revision_2/DEG/Table_S7_Mural_cell_type_markers.csv")

table(all_markers_major$cluster)

In [ ]:
# all.markers <- FindAllMarkers(object = obj_oi, 
#                               assay = "RNA",
#                               features = genes,
#                               only.pos = FALSE, 
#                               min.pct = 0.25
#                               )
# all.markers %>% 
#     group_by(cluster) %>% 
#     filter(p_val_adj < 0.05, avg_log2FC > 0.20) %>%
#     top_n(n = 5, wt = avg_log2FC) -> top

# all.markers %>% 
#     group_by(cluster) %>% 
#     filter(p_val_adj < 0.05, avg_log2FC > 0) -> sig

In [ ]:
## scale the gene expression of seurat object data for heatmap plot
obj_oi <- ScaleData(object = obj_oi, features = genes)

In [ ]:
options(repr.plot.height = 4,repr.plot.width = 6)
p1 <-  DotPlot(obj_oi,cols = viridis(3, option = "magma"),
             features = unique(top$gene)) + RotatedAxis(180) +
                                    theme(axis.title = element_blank(),legend.position = "top")
p1

## Draw upset plot for the significant genes across all comparisons

In [ ]:
## get the significant genes
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 1) %>%
  group_by(cluster) 

## check number of significant genes in each cluster
table(sig_genes_list$cluster)

In [ ]:
## Draw upset plot for the significant genes across clusters
library(UpSetR)
options(repr.plot.height = 6,repr.plot.width = 12)

## prepare the data for upset plot
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 1) %>%
  # dplyr::filter(logFC < -0.5) %>%
  group_by(cluster) 
sig_genes_list = split(sig_genes_list$genes, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
# length(sig_genes_list)

## draw the upset plot and add title "Number of upregulated DEGs in each region cluster"
# pdf(file = "./Results/Revision_2/DEG/Fibroblast_sig_genes_upset_plot.pdf",width = 12,height = 6)
upset(
  fromList(sig_genes_list), 
  order.by = "freq", 
  nsets = length(sig_genes_list), 
  nintersects =NA,
  main.bar.color = c("#5A8E60","#A26DC2","#0f0f0f","#C29DFF","#A26DC2", "#6A0DAD","#0f0f0f","#0f0f0f"),
  sets.bar.color = c("#5A8E60","#C29DFF", "#A26DC2", "#6A0DAD"),
  text.scale = c(2, 2, 2, 2, 3, 2),
  point.size = 5, line.size = 1.6, mb.ratio = c(0.6, 0.4),
  att.pos = "bottom",
  mainbar.y.label = "Number of upregulated DEGs",
  sets.x.label = "Region Clusters"
  )
# dev.off()

  # "SMC_1" = "#6A0DAD",   # strong violet
  # "SMC_2" = "#A26DC2",   # medium purple
  # "SMC_3" = "#",
  # "Pericyte" = "#5A8E60"

##  Draw bubble plot

In [ ]:
# Normalize the data
obj_oi <- NormalizeData(obj_oi)

# First check what levels you have
levels(factor(obj_oi$Cell_type))

In [ ]:
# ── Option A: Custom manual order ────────────────────────────────────────────
cell_order <- rev(c("Pericyte", "SMC_1", "SMC_2","SMC_3"))   # replace with your actual cell type names
Idents(obj_oi) <- factor(obj_oi$Cell_type, levels = cell_order)

In [ ]:
# ── 1. Top upregulated DEGs per cluster ───────────────────────────────────
res %>%
  filter(logFC > 0, FDR < 0.05) %>%
  group_by(cluster) -> top

# Genes that appear in MORE than one cluster's top list (the "shared" ones)
shared_genes <- top %>%
  group_by(genes) %>%
  summarise(n_clusters = n_distinct(cluster), .groups = "drop") %>%
  filter(n_clusters > 1) %>%
  pull(genes)

# ── 2. Unique DEGs: significant, NOT in the shared top gene pool ──────────────
res %>%
  filter(logFC > 1, FDR < 0.05) %>%
  filter(!genes %in% shared_genes) %>%          # exclude shared top genes
  group_by(cluster) %>%
  arrange(FDR) %>%
  slice_head(n = 5) -> top_unique               # top 5 truly unique per cluster
# -- shared DEGs
res %>%
  filter(logFC > 1, FDR < 0.05) %>%
  group_by(cluster) %>%
  arrange(FDR) %>%
  slice_head(n = 5) -> top

# ── 3. Combine ────────────────────────────────────────────────────────────────
top$gene_type       <- "Shared"
top_unique$gene_type <- "Unique"

combined <- bind_rows(top, top_unique) %>%
  distinct(genes, .keep_all = TRUE) %>%        # deduplicate by gene only
  arrange(cluster, logFC)

all_genes  <- unique(combined$genes)

gene_types <- combined %>%
  distinct(genes, gene_type) %>%
  group_by(genes) %>%
  summarise(gene_type = if_else(any(gene_type == "Shared"), "Shared", "Unique"),
            .groups = "drop") %>%
  arrange(match(genes, all_genes))

# df_all <- res

# res %>%
#   filter(logFC > 1, FDR < 0.05) %>%
#   group_by(cluster) %>%
#   arrange(FDR) %>%
#   slice_head(n = 5) -> top

In [ ]:
## remove unrelated genes
g_remove = c('COLEC12', 'PLXDC2', 'ALCAM', 'ERBB4', 'EPB41L3', 'PCDH7', 'UNC13C', 'CACNA1A', 'NTM', 'TMEM108', 'RNF144A', 'ZNF331', 'HSP90AB1', 'JUN', 'RPLP1', 'SLC19A1', 'USP9Y')
genes_keep = setdiff(all_genes, g_remove)
genes_keep

In [ ]:
options(repr.plot.height = 4,repr.plot.width = 10)
p1 <-  DotPlot(obj_oi,cols = viridis(3, option = "magma"),
             features = unique(genes_keep)) + RotatedAxis(180) +
                                    theme(axis.title = element_blank(),legend.position = "top")
p1
ggsave(p1, file = "./Results/Revision_2/DEG/Figure_4x_Mural_cell_type_markers.pdf",width = 8, height = 3)

In [ ]:
### Making sure get the corrected layer and matrix
DefaultAssay(obj_oi) = "RNA"

### Aggregation to pseudobulk 
mtx = AggregateExpression(
        obj_oi, 
        # features = gene_oi,
        group.by = c("region_cluster","individualID"),
        assays = 'RNA',
        slot = "counts"
    ) 
mtx <- as.matrix(mtx$RNA)
## Get the library size for each sample
lib_size <- Matrix::colSums(mtx)

### Get the logCPM from the matrix
cpm <- t(t(mtx) / lib_size) * 1e6
logCPM <- log2(cpm + 1)
logCPM = logCPM[gene_types$genes,]

### Calculate the averaged logCPM
# extract region (everything before first "_")
region <- sub("_.*$", "", colnames(logCPM))

# sanity check
table(region)

logCPM_region <- sapply(
  split(seq_len(ncol(logCPM)), region),
  function(i) {
    rowMeans(logCPM[, i, drop = FALSE])
  }
)
logCPM_region_z <- t(scale(t(logCPM_region)))

In [ ]:
# ── 4. Build expression matrix ───────────────────────────────────────────────
# mat <- matrix(0,
#               nrow = length(all_genes),
#               ncol = length(unique(df_all$cluster)),
#               dimnames = list(all_genes, unique(df_all$cluster)))

# for (j in seq_along(all_genes)) {
#   for (i in seq_along(colnames(mat))) {
#     sub <- df_all[df_all$genes   == all_genes[j] &
#                   df_all$cluster == colnames(mat)[i], ]
#     if (nrow(sub) > 0) mat[j, i] <- sub$logFC
#   }
# }
mat = logCPM_region_z

# ── 5. Build significance matrix ─────────────────────────────────────────────
sig_mat <- matrix(1,
                  nrow = length(all_genes),
                  ncol = length(unique(df_all$cluster)),
                  dimnames = list(all_genes, unique(df_all$cluster)))

for (j in seq_along(all_genes)) {
  for (i in seq_along(colnames(sig_mat))) {
    sub <- df_all[df_all$genes   == all_genes[j] &
                  df_all$cluster == colnames(sig_mat)[i], ]
    if (nrow(sub) > 0) sig_mat[j, i] <- sub$FDR
  }
}

# ── 6. Reorder columns by cluster number ─────────────────────────────────────
col_order <- order(as.numeric(str_split_fixed(colnames(mat), "_", 2)[, 1]))
mat     <- mat[, col_order]
sig_mat <- sig_mat[, col_order]

# ── 7. Row annotation: Top 10 vs Unique ──────────────────────────────────────
row_anno_df <- data.frame(
  Type = gene_types$gene_type,
  row.names = gene_types$genes
)
row_anno_df <- row_anno_df[rownames(mat), , drop = FALSE]  # align order

row_ha <- rowAnnotation(
  Type = row_anno_df$Type,
  col  = list(Type = c("Shared" = "#E07B39",   # warm orange
                        "Unique" = "#4B8BBE")),  # steel blue
  annotation_legend_param = list(
    Type = list(title = "Gene type", title_gp = gpar(fontsize = 10))
  ),
  width = unit(3, "mm"),
  show_annotation_name = FALSE
)

In [ ]:
# ── 8. Draw heatmap ──────────────────────────────────────────────────────────

ht <- Heatmap(
  mat,
  cluster_rows    = FALSE,
  cluster_columns = FALSE,
  col = colorRamp2(c(-1,0,1), c("#5387be","#F7F7F7","#b31f2c")),
  # right_annotation = row_ha,
  show_column_names = TRUE,
  show_row_names    = TRUE,
  column_title      = NULL,
  row_names_side    = "left",
  row_title_gp      = gpar(fontsize = 10),
  row_names_gp      = gpar(fontsize = 8),
  row_names_max_width = max_text_width(rownames(mat), gp = gpar(fontsize = 12)),
  cell_fun = function(j, i, x, y, w, h, fill) {
    if (sig_mat[i, j] < 0.05)
      grid.text("*", x, y, gp = gpar(fontsize = 18, col = "black"), vjust = "center")
  }
)
pdf("./Results/Revision_2/DEG/Figure_2X_Endothelial_DEG_heatmap_wC5.pdf",width = 3,height = 6)
draw(ht)
dev.off()


### Perform GO enrichment analysis for the significant genes in each cluster


In [ ]:
suppressPackageStartupMessages({
library(clusterProfiler)
library('org.Hs.eg.db')
# library(ReactomePA)
})

In [ ]:
res = read.csv("./Results/Revision_2/DEG/Mural_CT_edgeR_sig.csv",row.names = 1)
res_sig = res %>% dplyr::filter(FDR < 0.05) %>% dplyr::filter(logFC > 0.5)
table(res_sig$cluster)

In [ ]:
#########################################################################
#### find enriched GOBP for consistent signals in inhibitory neurons ####
#########################################################################
res$gene_id <- mapIds(
  # Replace with annotation package for the organism relevant to your data
  org.Hs.eg.db,
  # The vector of gene identifiers we want to map
  keys = res$genes,
  # Replace with the type of gene identifiers in your data
  keytype = "SYMBOL",
  # Replace with the type of gene identifiers you would like to map to
  column = "ENTREZID",
  multiVals = "first"
)

In [ ]:
### Run enrichment analysis for overlapped upregulated genes in C1, C2, and C3 clusters
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  group_by(cluster)
sig_genes_list = split(sig_genes_list$gene_id, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
length(sig_genes_list)

In [ ]:
ego_result = data.frame()

for (i in unique(res$cluster)){
    cluster_genes <- res %>%
        dplyr::filter(cluster == i, logFC > 0.5, FDR < 0.05) %>%
        pull(gene_id) %>%
        unique()
    print(length(cluster_genes))

    ego <- enrichGO(gene = cluster_genes,
                OrgDb = org.Hs.eg.db,
                keyType = "ENTREZID",
                ont = "BP",
                pAdjustMethod = "BH",
                readable      = TRUE,
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.05)
                
    if (is.null(ego)){
            print(paste("No significant results for",i))
        }else{
            ## remove redundancy
            ego_df = ego@result
            ego_df$cluster = i
        }
    ego_result <- rbind(ego_result, ego_df)
}

In [ ]:
##  Draw the heatmap top 5 significant pathways for each cluster and save the plot
top_pathways <- ego_result %>%
  dplyr::filter(p.adjust < 0.05) %>%
  group_by(cluster) %>%
  arrange(p.adjust) %>%
  slice_head(n = 5)
top_pathways <- top_pathways %>%
  ungroup() %>%
  mutate(cluster = factor(cluster, levels = c("Pericyte","SMC-1","SMC-2","SMC-3","SMC-4")))

options(repr.plot.width = 10.5,repr.plot.height = 9)
p3 <- ggplot(top_pathways, aes(x = cluster, y = Description, fill = -log10(p.adjust))) +
  geom_tile() +
  scale_fill_viridis(begin = 0,end = 1,option = "viridis") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1,size = 18),
        axis.text.y = element_text(size = 18))
p3

In [ ]:
##  Draw the heatmap top 5 significant pathways for each cluster and save the plot
top_pathways <- ego_result %>%
  dplyr::filter(p.adjust < 0.05) %>%
  group_by(cluster) %>%
  arrange(p.adjust) %>%
  slice_head(n = 5)

top_pathways <- top_pathways %>%
  ungroup() %>%
  mutate(
    cluster = factor(cluster, levels = c("Pericyte","SMC-1","SMC-2","SMC-3","SMC-4")),
    Description_wrapped = str_wrap(Description, width = 25)
  ) %>%
  # sort the data frame by cluster then FDR before creating the factor
  arrange(cluster, p.adjust) %>%
  mutate(
    Description_wrapped = factor(Description_wrapped, 
                                 levels = rev(unique(Description_wrapped)))
  )

options(repr.plot.width = 10, repr.plot.height = 10)

p3 <- ggplot(top_pathways, aes(x = cluster, y = Description_wrapped, 
                                fill = -log10(p.adjust))) +
  geom_tile(color = "white", linewidth = 0.4) +
  scale_fill_gradientn(
    colours = c("#F5F5F5", "#D6604D", "#8B0000"),   # white → coral → dark red
    name    = expression(-log[10](p.adjust))
  ) +
  scale_y_discrete(labels = function(x) x) +         # keeps \n line breaks intact
  labs(x = NULL, y = NULL) +
  theme_classic() +
  theme(
    axis.text.x  = element_text(angle = 45, hjust = 1, size = 10),
    axis.text.y  = element_text(size = 8, lineheight = 0.85),  # tighten wrapped lines
    legend.title = element_text(size = 9),
    legend.text  = element_text(size = 6),
    panel.border = element_rect(color = "grey80", fill = NA, linewidth = 0.5)
  )
p3

## Cell type composition across brain regions

In [ ]:
## get meta data to work on
meta <- obj_oi@meta.data
meta$brain_region = paste(meta$regionID, str_to_title(str_trim(meta$region_name)),sep = "_")

## get the count table
counts <- meta %>%
  group_by(Cell_type, brain_region) %>%
  summarise(freq = n(), .groups = "drop")
colnames(counts) <- c("Cell_type","brain_region","freq")

## reorder the brain region based on the number code
counts$order <- as.numeric(str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,1])
counts <- counts[order(counts$Cell_type),]
counts <- counts[order(counts$order),]

## making sure corrected order
counts$brain_region_plot = str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,2]
counts$brain_region_plot<- factor(counts$brain_region_plot, levels=unique(counts$brain_region_plot))


counts$Cell_type <- factor(counts$Cell_type)

col1=c(
  "SMC_1" = "#6A0DAD",   # strong violet
  "SMC_2" = "#A26DC2",   # medium purple
  "SMC_3" = "#C29DFF",
  "Pericyte" = "#5A8E60"
)

counts$color <- col1[counts$Cell_type]

In [ ]:
p4 <- ggplot(counts, aes(fill=Cell_type, y=freq, x=brain_region_plot)) + 
        geom_bar(position="fill", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 20),
              axis.text.y = element_text(size = 20),
              axis.title.y = element_text(size = 20)) +
        scale_fill_manual(values=col1) +
        xlab("") +
        ylab("Frequency")
options(repr.plot.width = 12, repr.plot.height = 6.5, repr.plot.res = 100) # set the resolution
# p3
p4
ggsave(filename = "./Results/Revision_2/Figures/Figure_4X_Mural_Cell_type_region_proportion.pdf", 
patchwork::wrap_plots(p4, ncol = 1),
scale = 1, width = 12, height = 7.5)

In [ ]:
sessionInfo()